# Vienna Stay Assistant: run the data pipeline

Open this notebook in VS Code or Jupyter, select a **Python 3.11** kernel, start Docker Desktop, and choose **Run All**.
Install `requirements.txt` first (see README), then select that environment as the notebook kernel.
The notebook starts databases, downloads source files, then loads vectors and PostgreSQL data.
First-time setup requires internet access. Full embedding can take a long time on CPU.


## 1. Configuration

Keep `LIMIT = None` for full imports. Use `LIMIT = 10` for samples in both databases. The vector sample uses a separate collection; the SQL sample contains the first N listings, up to N candidate reviews belonging to them, all their calendar rows, and all districts.

`SQL_REPLACE = False` protects existing SQL data. Set it to `True` only when you intend to replace the five accommodation tables, including when moving from a sample to a full import. SQL ingestion history and Qdrant collections are preserved.


In [1]:
LIMIT = 500
SQL_REPLACE = True  # Set True to explicitly replace an existing SQL dataset.

In [2]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

if sys.version_info < (3, 11):
    raise RuntimeError("Select a Python 3.11 or newer notebook kernel.")

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data_setup" / "download_data.py").is_file() and (p / "data_setup" / "load_vectors.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the ViennaStayAssistant project directory.")

PYTHON = Path(sys.executable)  # Use the selected notebook kernel environment.
DEVICE = "auto"  # CUDA when available, otherwise CPU; set "cpu" or "cuda" to override.

def run(*command):
    # Stream script output into the notebook and stop on the first failed command.
    command = [str(part) for part in command]
    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", env={**os.environ, "PYTHONIOENCODING": "utf-8"}) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            returncode = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)

print(f"Project: {ROOT}")


Project: c:\Users\hausb\OneDrive\Desktop\FH\Special Chapters of Applied AI\SCAAI_group_G


## 2. Start databases

Docker Desktop must be running. Wait for the services and the Qdrant API before continuing.


In [3]:
import time
from urllib.error import URLError
from urllib.request import urlopen

if shutil.which("docker") is None:
    raise RuntimeError("Install Docker Desktop with Docker Compose, then restart the notebook. Or run docker compose up manually")
run("docker", "compose", "up", "--wait", "--wait-timeout", "120", "postgres", "qdrant")

deadline = time.monotonic() + 60
while True:
    try:
        with urlopen("http://localhost:6353/collections", timeout=3) as response:
            if response.status == 200:
                break
    except (URLError, TimeoutError, OSError):
        pass
    if time.monotonic() >= deadline:
        raise RuntimeError("Qdrant did not become ready. Check Docker Desktop and rerun this cell.")
    time.sleep(1)
print("Databases are ready.")

 Container vienna-stay-assistant-qdrant-1  Running
 Container vienna-stay-assistant-postgres-1  Running
 Container vienna-stay-assistant-postgres-1  Waiting
 Container vienna-stay-assistant-qdrant-1  Waiting
 Container vienna-stay-assistant-qdrant-1  Healthy
 Container vienna-stay-assistant-postgres-1  Healthy
Databases are ready.


## 3. Download source data

Runs `data_setup/download_data.py` for the **2026-09-17** snapshot. Existing nonempty files are kept.


In [4]:
run(PYTHON, "-u", "-m", "data_setup.download_data")

Skipping existing file: C:\Users\hausb\OneDrive\Desktop\FH\Special Chapters of Applied AI\SCAAI_group_G\data\listings.csv
Skipping existing file: C:\Users\hausb\OneDrive\Desktop\FH\Special Chapters of Applied AI\SCAAI_group_G\data\calendar.csv
Skipping existing file: C:\Users\hausb\OneDrive\Desktop\FH\Special Chapters of Applied AI\SCAAI_group_G\data\reviews.csv
Skipping existing file: C:\Users\hausb\OneDrive\Desktop\FH\Special Chapters of Applied AI\SCAAI_group_G\data\neighbourhoods.csv
Skipping existing file: C:\Users\hausb\OneDrive\Desktop\FH\Special Chapters of Applied AI\SCAAI_group_G\data\neighbourhoods.geojson
All five data files are available in C:\Users\hausb\OneDrive\Desktop\FH\Special Chapters of Applied AI\SCAAI_group_G\data.


## 4. Load vectors

Runs `data_setup/load_vectors.py`: PII masking, multilingual embedding, and Qdrant upload.
The default review window is **2024-09-17 through 2026-09-17**.
Identical reruns resume partial uploads. A successful load prints its collection and writes a completion manifest to `derived/vectors/`.


In [5]:
command = [PYTHON, "-u", "-m", "data_setup.load_vectors", "--device", DEVICE]
if LIMIT is not None:
    if not isinstance(LIMIT, int) or LIMIT <= 0:
        raise ValueError("LIMIT must be None or a positive integer.")
    command.extend(["--limit", str(LIMIT)])
run(*command)

Checking source files and Qdrant connection...
Loading Presidio and German/English spaCy models...
Loading multilingual embedding model on cpu...

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 18753.69it/s]
c:\Users\hausb\OneDrive\Desktop\FH\Special Chapters of Applied AI\SCAAI_group_G\data_setup\load_vectors.py:324: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  size = model.get_sentence_embedding_dimension()
{"collection": "vienna_2026_09_17_c451f820da153186", "status": "loading"}
{"collection": "vienna_2026_09_17_c451f820da153186", "conflicting_review_ids": 376, "listings_read": 16, "documents_accepted": 16, "chunks_already_present": 0, "chunks_uploaded": 16}
{"collection": "vienna_2026_09_17_c451f820da153186", "conflicting_review_ids": 376, "listings_read": 32, "documents_accepted": 32, "chunks_already_present": 0, "chunks_uploaded": 32}
{"collection": "vienna_2026_09_17_c451f820da153186", "conflicting_review_

## 5. Load PostgreSQL/PostGIS

Runs `data_setup/load_sql.py` using the selected kernel environment and the shared `LIMIT`. Install the updated `requirements.txt` first, including the PostgreSQL driver.

The loader uses the `vienna_stay_assistant` schema inside the `vienna_stay` database, masks free text, and loads districts, hosts, listings, recent reviews, and calendar in one transaction. The default review window is **2024-09-17 through 2026-09-17**, matching the vector loader. Failures roll back the import; populated SQL tables require `SQL_REPLACE = True`. Full privacy processing may take time.

Connection defaults match Compose; `DATABASE_URL` or standard `PG*` environment variables override them. Successful loads save validation reports to `derived/sql/` and `ingestion_runs`. After running configuration/startup cells, you can run this section independently when vectors are already loaded.


In [6]:
if not isinstance(SQL_REPLACE, bool):
    raise ValueError("SQL_REPLACE must be True or False.")
sql_command = [PYTHON, "-u", "-m", "data_setup.load_sql"]
if LIMIT is not None:
    if not isinstance(LIMIT, int) or LIMIT <= 0:
        raise ValueError("LIMIT must be None or a positive integer.")
    sql_command.extend(["--limit", str(LIMIT)])
if SQL_REPLACE:
    sql_command.append("--replace")
run(*sql_command)

Connected. Loading German/English privacy models (no embedding model)...
calendar: 10000 rows prepared
calendar: 20000 rows prepared
calendar: 30000 rows prepared
calendar: 40000 rows prepared
calendar: 50000 rows prepared
calendar: 60000 rows prepared
calendar: 70000 rows prepared
calendar: 80000 rows prepared
calendar: 90000 rows prepared
calendar: 100000 rows prepared
Complete (sample). Validation report: C:\Users\hausb\OneDrive\Desktop\FH\Special Chapters of Applied AI\SCAAI_group_G\derived\sql\dcc90994-ed38-4710-8d1e-316edc9aebc0.json


Databases remain running. When finished, run `docker compose down` in a terminal to stop them while keeping data.

Data: [Inside Airbnb](https://insideairbnb.com/get-the-data/) · [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
Startup uses Docker Compose's [wait option](https://docs.docker.com/reference/cli/docker/compose/up/).
